# Qwen3.6-27B-FP8 — evidence-first RAW OCR diagnostic
## 1. Objective
Run **locally and offline in Domino on one H100**, using only the supplied checkpoint. Establish correct image transcription before scaling. This notebook deliberately stops on unsupported architecture, missing visual tensors, incompatible quantization, unexpected placement, or unhealthy validation.

**Run cells in order.** Initial execution performs one measured page, a separate short warmup, a controlled blank-image ablation, and a small resolution comparison only when earlier checks are healthy. Batch processing is locked by default. Inspect the displayed page/output, enter the review evidence in section 22, and rerun that cell before later benchmarks. No installation, model download, external OCR, or external inference occurs.

This is a new diagnostic implementation; the failed code, local checkpoint, and H100 are not available to its author. It cannot truthfully declare the historical root cause or guarantee local model support in advance. Unsupported runtime combinations produce actionable evidence instead of fabricated APIs.

## 2. Failure symptoms
Reported: >3 hours / ~50 pages and repeated punctuation such as `!!!!!!!!`. These are hypotheses to investigate, not proof of CPU execution, broken FP8, or missing images. A 50-page estimate here is an extrapolation from observed pages, not a service-level promise. Preserve original output, including pathological text.


## 3. Environment inspection
Inspect versions before any changes. Missing required packages stop execution and list offline installation commands using a locally supplied wheelhouse. Do not upgrade Domino's CUDA stack automatically. Restart the kernel after any administrator-approved package change. `flash_attn`, OpenCV, and `flash-linear-attention` are optional; their presence alone proves no kernel is active.

In [ ]:
import os, sys, json, time, re, csv, random, inspect, hashlib, traceback, subprocess, zipfile, stat, math
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter
from importlib import metadata, util
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
PACKAGES = ["torch", "transformers", "accelerate", "PyMuPDF", "Pillow", "numpy", "safetensors", "jinja2", "flash_attn", "flash-linear-attention", "triton", "kernels"]
VERSIONS = {}
for p in PACKAGES:
    try: VERSIONS[p] = metadata.version(p)
    except metadata.PackageNotFoundError: VERSIONS[p] = None
print(json.dumps({"Python":sys.version, **VERSIONS}, indent=2))
required = ["torch", "transformers", "accelerate", "PyMuPDF", "Pillow", "numpy", "safetensors", "jinja2"]
missing = [p for p in required if VERSIONS[p] is None]
if missing:
    raise RuntimeError("Missing packages: " + str(missing) +
        "\nInstall only these, offline: python -m pip install --no-index --find-links=/path/to/wheelhouse " + " ".join(missing))
import torch, transformers, numpy as np
from PIL import Image, ImageOps, ImageEnhance, ImageFilter, ImageDraw
from IPython.display import display
try:
    import pymupdf
except ImportError as exc:
    raise RuntimeError("PyMuPDF is installed but its pymupdf import is unavailable. Use a compatible PyMuPDF wheel; do not install the unrelated fitz package.") from exc
print("Torch CUDA build:", torch.version.cuda, "CUDA available:", torch.cuda.is_available())

## 4. Configuration
Start with a 1,400-pixel long edge and 768 output tokens. These are conservative diagnostic settings, not an accuracy claim. Sampling settings are recorded but not forwarded when sampling is disabled. A cooperative time limit is checked between decoding steps; it cannot interrupt a stalled CUDA kernel. `MAX_QWEN_CALLS` includes diagnostics, warmup, ablations, and batch calls. Full-dataset processing requires explicitly removing page/customer limits and raising that budget.

`VALIDATION_PDF` and the 1-based `VALIDATION_PAGE` let you choose a legible page after viewing the automatic selection. Each run gets its own output directory. A configuration fingerprint invalidates validation after changes.

In [ ]:
@dataclass
class Config:
    ZIP_PATH: str = "kyc_documents.zip"
    MODEL_PATH: str = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main"
    WORK_DIR: str = "kyc_qwen36_work"
    OUTPUT_DIR: str = "kyc_qwen36_outputs"
    REQUIRED_DOCUMENTS: tuple = ("JUSTIFICATIF IDENTITE.PDF", "JUSTIFICATIF DOMICILE.PDF", "CONVENTION COMPTE.PDF", "FATCA.PDF", "CARTON SIGNATUTE.PDF")
    RENDER_DPI: int = 150
    MAX_IMAGE_LONG_EDGE: int = 1400
    MIN_IMAGE_LONG_EDGE: int = 0  # never upscale by default
    MAX_RENDER_PIXELS: int = 16_000_000
    MAX_NEW_TOKENS: int = 768
    DO_SAMPLE: bool = False
    TEMPERATURE: object = None
    TOP_P: object = None
    RANDOM_SEED: int = 42
    MAX_CUSTOMERS: object = 1
    MAX_PAGES: object = 50
    MAX_QWEN_CALLS: int = 65
    BENCHMARK_LEVEL: int = 1  # 1=one, 2=three, 3=ten, 4=fifty, 5=full
    ENABLE_PREPROCESSING: bool = False
    ROTATE_DEGREES: int = 0  # manual orientation, no unsupported automatic detector
    DESKEW_DEGREES: float = 0.0  # measured/manual, conservative +/-5 degrees
    CROP_MARGINS: bool = False
    CROP_WHITE_THRESHOLD: int = 248
    CONTRAST: float = 1.0
    GRAYSCALE: bool = False
    DENOISE: bool = False
    SHARPEN: bool = False
    ENABLE_BATCHING: bool = False
    BATCH_SIZE: int = 1
    DEGENERACY_THRESHOLD: float = 0.80
    MAX_PAGE_SECONDS: float = 120.0
    MAX_GENERATION_SECONDS: float = 90.0
    MAX_ERROR_RATE: float = 0.0
    MAX_LIVE_MEMORY_GROWTH_GIB: float = 2.0
    MIN_FREE_VRAM_GIB: float = 6.0
    MAX_INPUT_TOKENS: int = 12000
    GPU_INDEX: int = 0
    CUSTOMER_ID: object = None
    VALIDATION_PDF: object = None
    VALIDATION_PAGE: int = 1
    RUN_RESOLUTION_BENCHMARK: bool = True
    RESOLUTION_LONG_EDGES: tuple = (1000, 1800)
    PROFILE_FIRST_PAGE: bool = True
    TRUST_LOCAL_CODE: bool = False
    HASH_WEIGHT_FILES: bool = False  # potentially slow on a network mount
    MAX_ZIP_UNCOMPRESSED_GIB: float = 20.0
CFG = Config()
assert not CFG.DO_SAMPLE, "This baseline requires deterministic decoding."
assert 1 <= CFG.MAX_NEW_TOKENS <= 2048
assert CFG.BENCHMARK_LEVEL in range(1,6)
random.seed(CFG.RANDOM_SEED); np.random.seed(CFG.RANDOM_SEED); torch.manual_seed(CFG.RANDOM_SEED)
RUN_ID = time.strftime("%Y%m%d_%H%M%S") + "_" + os.urandom(3).hex()
OUT = Path(CFG.OUTPUT_DIR).resolve()/RUN_ID; OUT.mkdir(parents=True, exist_ok=False)
WORK = Path(CFG.WORK_DIR).resolve(); WORK.mkdir(parents=True, exist_ok=True)
EVIDENCE = {"run_id":RUN_ID, "versions":VERSIONS, "historical_root_cause":"Not observable from this environment alone"}
RESULTS, ERRORS, BENCHMARKS = [], [], []
CALLS = 0
SINGLE_PAGE_VALIDATION_PASSED = False
VALIDATED_FINGERPRINT = None
PASSED_LEVELS = set()
class ConfigurationError(RuntimeError): pass
class SafetyStop(RuntimeError): pass
def persist_evidence():
    (OUT/"diagnostic_evidence.json").write_text(json.dumps(EVIDENCE, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
def stop(message):
    EVIDENCE["fatal_error"] = message; persist_evidence()
    report={"observed_failure":message,"historical_cause":"Unproven; this is an observed failure in the current run."}
    (OUT/"root_cause_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    (OUT/"root_cause_report.md").write_text("# Diagnostic stopped\n\n"+message+"\n\nHistorical causality remains unproven.",encoding="utf-8")
    raise ConfigurationError(message)
def fingerprint():
    keys = ["MODEL_PATH","RENDER_DPI","MAX_IMAGE_LONG_EDGE","MAX_NEW_TOKENS","ENABLE_PREPROCESSING","ROTATE_DEGREES","DESKEW_DEGREES","CROP_MARGINS","CROP_WHITE_THRESHOLD","CONTRAST","GRAYSCALE","DENOISE","SHARPEN","MAX_INPUT_TOKENS","GPU_INDEX"]
    return hashlib.sha256(json.dumps({k:getattr(CFG,k) for k in keys},sort_keys=True).encode()).hexdigest()
(OUT/"configuration.json").write_text(json.dumps(asdict(CFG), indent=2), encoding="utf-8")
print("Run outputs:", OUT)

## 5. Local model inspection
Inspect all local configuration files, templates, weight indexes, and Safetensors headers before loading. Header reads do not load large weight tensors. Missing shards, invalid offsets, and vocab inconsistencies stop the pipeline. Optional full-file hashes establish reproducibility, **not** proof of correctness without trusted reference hashes. The directory label is never treated as architecture evidence.

In [ ]:
MODEL_PATH = Path(CFG.MODEL_PATH)
if not MODEL_PATH.is_dir(): stop(f"Local model directory does not exist: {MODEL_PATH}")
def read_json(path):
    try: return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc: stop(f"Cannot parse {path.name}: {type(exc).__name__}: {exc}")
LOCAL = {p.name: read_json(p) for p in MODEL_PATH.glob("*.json") if not p.name.endswith(".index.json") and p.name != "tokenizer.json"}
if "config.json" not in LOCAL: stop("config.json is missing")
RAW_CONFIG = LOCAL["config.json"]
for name in ("config.json","preprocessor_config.json","processor_config.json","tokenizer_config.json","generation_config.json","special_tokens_map.json"):
    obj = LOCAL.get(name)
    if obj is not None:
        print("\n" + name)
        # Added vocab can be huge; retain full file on disk, display a bounded overview.
        print(json.dumps(obj, ensure_ascii=False, indent=2)[:16000])
TEMPLATES = {str(p.relative_to(MODEL_PATH)):p.read_text(encoding="utf-8") for p in MODEL_PATH.rglob("*.jinja")}
print("Local templates:", list(TEMPLATES))
print("Local Python source:", [p.name for p in MODEL_PATH.glob("*.py")])
WEIGHT_META = {}; WEIGHT_BYTES = 0
shards = sorted(MODEL_PATH.glob("*.safetensors"))
indexes = list(MODEL_PATH.glob("*.safetensors.index.json"))
for index in indexes:
    wm = read_json(index).get("weight_map", {})
    absent = [f for f in set(wm.values()) if not (MODEL_PATH/f).is_file()]
    if absent: stop("Missing weight shards: " + str(absent))
if not shards: stop("No Safetensors weights found. This diagnostic refuses uninspected pickle weights.")
for shard in shards:
    size=shard.stat().st_size; WEIGHT_BYTES += size
    try:
        with shard.open("rb") as f:
            header_len = int.from_bytes(f.read(8), "little")
            if not 0 < header_len < min(size, 100_000_000): stop("Invalid Safetensors header: " + shard.name)
            header=json.loads(f.read(header_len))
        for name, meta in header.items():
            if name == "__metadata__": continue
            a,b=meta["data_offsets"]
            if not 0 <= a <= b <= size-8-header_len: stop("Invalid weight offsets: " + name)
            if name in WEIGHT_META: stop("Duplicate tensor across shards: " + name)
            WEIGHT_META[name] = {"dtype":meta["dtype"],"shape":meta["shape"],"file":shard.name}
    except ConfigurationError: raise
    except Exception as exc: stop(f"Weight header failure {shard}: {exc}")
for index in indexes:
    wm=read_json(index).get("weight_map",{})
    mismatched=[k for k,v in wm.items() if k not in WEIGHT_META or WEIGHT_META[k]["file"] != v]
    if mismatched: stop("Weight index/header mismatch: " + str(mismatched[:10]))
print("Checkpoint size GiB:", round(WEIGHT_BYTES/2**30,2))
print("Checkpoint tensor dtypes:", Counter(v["dtype"] for v in WEIGHT_META.values()))
if CFG.HASH_WEIGHT_FILES:
    hashes={}
    for p in shards:
        h=hashlib.sha256()
        with p.open("rb") as f:
            for chunk in iter(lambda:f.read(16*1024*1024), b""): h.update(chunk)
        hashes[p.name]=h.hexdigest()
    EVIDENCE["weight_sha256"]=hashes
EVIDENCE["local_config"]=RAW_CONFIG
EVIDENCE["checkpoint_dtype_counts"]=dict(Counter(v["dtype"] for v in WEIGHT_META.values()))
persist_evidence()

## 6. Model architecture validation
Resolve `AutoConfig` from the installed Transformers release. Use registered image-to-text auto mappings first, with an exact locally declared Transformers architecture as a fallback. Local custom code is not executed unless `TRUST_LOCAL_CODE=True` after source review. Unsupported combinations stop; the notebook does not guess a Qwen class from a marketing name.

In [ ]:
try:
    hf_config = transformers.AutoConfig.from_pretrained(str(MODEL_PATH), local_files_only=True, trust_remote_code=CFG.TRUST_LOCAL_CODE)
except Exception as exc:
    stop(f"Installed Transformers cannot resolve this local configuration: {type(exc).__name__}: {exc}")
print("MODEL TYPE:", hf_config.model_type)
print("ARCHITECTURES:", getattr(hf_config,"architectures",None))
MODEL_LOADER = None
for auto_name in ("AutoModelForImageTextToText", "AutoModelForVision2Seq"):
    auto=getattr(transformers,auto_name,None)
    if auto is None: continue
    try:
        if type(hf_config) in auto._model_mapping:
            MODEL_LOADER=auto; break
    except (KeyError,ImportError,AttributeError) as exc:
        print("Auto mapping unavailable:", auto_name, str(exc)[:250])
if MODEL_LOADER is None:
    for name in getattr(hf_config,"architectures",[]) or []:
        candidate=getattr(transformers,name,None)
        if candidate is not None and getattr(candidate,"config_class",None) is type(hf_config):
            MODEL_LOADER=candidate; break
if MODEL_LOADER is None and CFG.TRUST_LOCAL_CODE:
    for name in ("AutoModelForImageTextToText","AutoModelForVision2Seq"):
        if name in (getattr(hf_config,"auto_map",{}) or {}) and hasattr(transformers,name):
            MODEL_LOADER=getattr(transformers,name); break
EVIDENCE["config_class"]=type(hf_config).__name__

## 7. Multimodal capability validation
Require an explicit vision configuration, a resolvable model class, then (later) an image processor and real vision tensors. An unknown architecture is reported as unsupported, not incorrectly labelled text-only.

In [ ]:
vision_config = RAW_CONFIG.get("vision_config") or RAW_CONFIG.get("visual_config")
if not vision_config:
    signals = json.dumps(RAW_CONFIG).lower()
    if not any(s in signals for s in ("vision", "image", "visual")):
        stop("THIS CHECKPOINT DOES NOT SUPPORT DIRECT IMAGE OCR. No image/vision capability is declared in config.json.")
    stop("Vision capability cannot be established from the installed configuration; checkpoint support is unresolved.")
if MODEL_LOADER is None: stop("No compatible multimodal class exists in installed Transformers for this checkpoint.")
print("MODEL LOADER:", MODEL_LOADER.__name__)
print("MULTIMODAL SUPPORT: declared in configuration; execution pending")
print("VISION SUPPORT:", vision_config)
EVIDENCE["multimodal_config"]=True

## 8. FP8 validation
Do not pass a second quantization configuration. The checkpoint's embedded metadata must identify FP8 and the installed quantizer must recognize it. Inspect header dtypes and scale tensors. After loading, inspect actual quantized modules, quantizer, parameter dtypes, and observed profiler operations. A module named FP8 alone does not prove an efficient hardware kernel.

In [ ]:
QCONFIG = RAW_CONFIG.get("quantization_config")
if not isinstance(QCONFIG,dict): stop("FP8 checkpoint label has no embedded quantization_config; refuse to invent or apply new quantization.")
print("Quantization metadata:", json.dumps(QCONFIG,indent=2))
method=str(QCONFIG.get("quant_method","")).lower()
fp8_metadata = any(s in json.dumps(QCONFIG).lower() for s in ("fp8","float8"))
fp8_weights = [k for k,v in WEIGHT_META.items() if "F8" in v["dtype"].upper() or "FLOAT8" in v["dtype"].upper()]
scale_weights = [k for k in WEIGHT_META if "scale" in k.lower()]
print("FP8 tensors:",len(fp8_weights),"Scale tensors:",len(scale_weights),"examples:",scale_weights[:8])
if not fp8_metadata: stop("Checkpoint quantization metadata does not establish FP8. Inspect checkpoint provenance.")
try:
    from transformers.quantizers.auto import AutoQuantizationConfig
    parsed_quantization = AutoQuantizationConfig.from_dict(QCONFIG)
except Exception as exc: stop(f"Installed Transformers does not support checkpoint quantization: {exc}")
EVIDENCE["fp8"]={"metadata":QCONFIG,"header_fp8_tensors":len(fp8_weights),"scale_tensors":len(scale_weights),"quantization_class":type(parsed_quantization).__name__,"kernel_status":"unverified before inference"}
print("No additional FineGrainedFP8Config or FP8Linear substitution will be applied.")

## 9. GPU/device diagnostics
Use one explicit CUDA device; do not silently offload to CPU. Weight-file size plus 20% and a configurable reserve provides only a **lower-bound planning estimate**; actual activation/KV memory may be larger. Fail on inadequate headroom. Other processes and H100 partitioning can affect available memory.

In [ ]:
if not torch.cuda.is_available(): stop("CUDA unavailable; refusing CPU OCR.")
DEVICE = torch.device(f"cuda:{CFG.GPU_INDEX}")
torch.cuda.set_device(DEVICE)
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i)
    print(i,p.name,"VRAM GiB",round(p.total_memory/2**30,2),"capability",torch.cuda.get_device_capability(i))
if "H100" not in torch.cuda.get_device_name(DEVICE): stop("Selected GPU is not an H100; check Domino hardware assignment.")
def gpu_snapshot():
    free,total=torch.cuda.mem_get_info(DEVICE)
    return {"allocated_gib":torch.cuda.memory_allocated(DEVICE)/2**30,"reserved_gib":torch.cuda.memory_reserved(DEVICE)/2**30,
            "peak_allocated_gib":torch.cuda.max_memory_allocated(DEVICE)/2**30,"peak_reserved_gib":torch.cuda.max_memory_reserved(DEVICE)/2**30,
            "free_gib":free/2**30,"total_gib":total/2**30}
def smi():
    try:
        return subprocess.run(["nvidia-smi","--query-gpu=index,name,utilization.gpu,memory.used,memory.total","--format=csv"],capture_output=True,text=True,timeout=10).stdout.strip()
    except Exception as exc: return f"nvidia-smi unavailable: {exc}"
print(smi()); print(gpu_snapshot())
free,_=torch.cuda.mem_get_info(DEVICE)
if free < WEIGHT_BYTES*1.2 + CFG.MIN_FREE_VRAM_GIB*2**30:
    stop("Insufficient estimated free VRAM for explicit single-GPU load; no CPU fallback is allowed.")

## 10. Model loading — once
Select FlashAttention 2 only when the installed class declares support and its package imports successfully; otherwise select SDPA if declared, then eager. Backend configuration is inspected again after loading; the profiler reports observed operations separately. Hybrid/linear attention may use additional model-specific kernels. No FlashAttention installation is attempted. Load errors retain the original exception and stop.

In [ ]:
resolved_class=MODEL_LOADER
if hasattr(MODEL_LOADER,"_model_mapping"):
    try: resolved_class=MODEL_LOADER._model_mapping[type(hf_config)]
    except Exception: pass
flash_available=False
try:
    import flash_attn
    flash_available=True
except (ImportError,OSError,RuntimeError): pass
if flash_available and getattr(resolved_class,"_supports_flash_attn_2",False): REQUESTED_ATTN="flash_attention_2"
elif getattr(resolved_class,"_supports_sdpa",False): REQUESTED_ATTN="sdpa"
else: REQUESTED_ATTN="eager"
if "model" in globals(): stop("Model already exists. Restart the kernel to change loading configuration; do not load twice.")
t0=time.perf_counter()
try:
    model=MODEL_LOADER.from_pretrained(str(MODEL_PATH), config=hf_config, local_files_only=True,
        trust_remote_code=CFG.TRUST_LOCAL_CODE, device_map={"":str(DEVICE)},
        torch_dtype=torch.bfloat16, attn_implementation=REQUESTED_ATTN,
        low_cpu_mem_usage=True, use_safetensors=True)
    model.eval()
    torch.cuda.synchronize(DEVICE)
except Exception as exc: stop(f"Model/FP8/backend load failed: {type(exc).__name__}: {exc}")
MODEL_LOAD_TIME=time.perf_counter()-t0
print("MODEL CLASS:",type(model).__name__,"MODEL LOAD TIME:",MODEL_LOAD_TIME)
def placement_report():
    counts=Counter(); bad=[]
    for name,p in model.named_parameters():
        counts[str(p.device)]+=p.numel()
        if p.device != DEVICE: bad.append((name,str(p.device),p.numel()))
    cpu_buffers=[(n,str(b.device),b.numel()) for n,b in model.named_buffers() if b.device != DEVICE and b.numel()>1024]
    return {"device_map":getattr(model,"hf_device_map",{}),"parameter_devices":dict(counts),"gpu_modules":[n for n,d in getattr(model,"hf_device_map",{}).items() if str(d).startswith("cuda") or isinstance(d,int)],"cpu_disk_modules":[n for n,d in getattr(model,"hf_device_map",{}).items() if str(d) in ("cpu","disk")],"offloaded_parameters":bad,"non_cuda_large_buffers":cpu_buffers}
PLACEMENT=placement_report(); print(json.dumps(PLACEMENT,indent=2,default=str)); print(gpu_snapshot())
if PLACEMENT["offloaded_parameters"] or PLACEMENT["non_cuda_large_buffers"]: stop("Unexpected CPU/meta/other-GPU tensors; explicit H100 placement failed.")
q=getattr(model,"hf_quantizer",None)
quantized_modules=[(name,type(m).__module__+"."+type(m).__name__) for name,m in model.named_modules() if any(x in (type(m).__module__+type(m).__name__).lower() for x in ("fp8","float8","compressed_tensors"))]
print("Quantizer:",type(q).__name__,"is_quantized:",getattr(model,"is_quantized",False))
print("Quantized module classes:",dict(Counter(c for _,c in quantized_modules)))
print("Parameter dtypes:",dict(Counter(str(p.dtype) for p in model.parameters())))
if q is None or not getattr(model,"is_quantized",False): stop("Embedded FP8 quantizer is not active after load.")
live_fp8=sum(p.numel() for p in model.parameters() if "float8" in str(p.dtype))
if fp8_weights and live_fp8==0 and not quantized_modules: stop("FP8 weights appear expanded without a recognized quantized module. Refuse unverified dequantization.")
EVIDENCE["fp8"].update({"quantizer":type(q).__name__,"modules":dict(Counter(c for _,c in quantized_modules)),"live_fp8_parameters":live_fp8})
EVIDENCE.update(model_class=type(model).__name__,placement=PLACEMENT,model_load_seconds=MODEL_LOAD_TIME)

## 11. Processor loading — once
Require a real image processor and the checkpoint's own tokenizer/template. Check vocab range against the language embedding table. Processor signatures and special-token mappings are printed. The explicit processor call later uses the rendered local template plus a PIL image; it never substitutes text describing an image.

In [ ]:
t0=time.perf_counter()
try:
    processor=transformers.AutoProcessor.from_pretrained(str(MODEL_PATH),local_files_only=True,trust_remote_code=CFG.TRUST_LOCAL_CODE)
except Exception as exc: stop(f"Processor load failed: {type(exc).__name__}: {exc}")
PROCESSOR_LOAD_TIME=time.perf_counter()-t0
tokenizer=getattr(processor,"tokenizer",None)
image_processor=getattr(processor,"image_processor",None)
if tokenizer is None or image_processor is None: stop("Loaded processor lacks tokenizer or image_processor; direct image OCR is unsupported.")
print("PROCESSOR CLASS:",type(processor).__name__,"LOAD TIME:",PROCESSOR_LOAD_TIME)
print("processor.__call__:",inspect.signature(processor.__call__))
print("apply_chat_template:",inspect.signature(processor.apply_chat_template))
print("image processor:",type(image_processor).__name__,inspect.signature(image_processor.preprocess))
print("Image processor config:", image_processor.to_dict())
print("Special tokens:",tokenizer.special_tokens_map)
print("EOS:",tokenizer.eos_token_id,"PAD:",tokenizer.pad_token_id)
print("Image/video tokens:", {k:v for k,v in tokenizer.get_vocab().items() if any(t in k.lower() for t in ("image","vision","video"))})
embedding_size=model.get_input_embeddings().num_embeddings
if max(tokenizer.get_vocab().values()) >= embedding_size: stop("Tokenizer IDs exceed input embedding size; tokenizer/model mismatch.")
EVIDENCE.update(processor_class=type(processor).__name__,processor_load_seconds=PROCESSOR_LOAD_TIME)
print("Local generation configuration:",model.generation_config.to_dict())
SOURCE_EVIDENCE={}
for label,obj in (("model_forward",model.forward),("processor_call",processor.__call__),
                  ("processor_template",processor.apply_chat_template),("quantizer",type(q))):
    try:
        source=inspect.getsource(obj)
        matches=[line.strip() for line in source.splitlines() if any(term in line.lower() for term in ("thinking","reasoning","fp8","float8","pixel","image","attn_implementation"))]
        SOURCE_EVIDENCE[label]={"file":inspect.getsourcefile(obj),"relevant_lines":matches[:60]}
    except (OSError,TypeError) as exc: SOURCE_EVIDENCE[label]={"source_unavailable":str(exc)}
print("Installed source inspection:",json.dumps(SOURCE_EVIDENCE,indent=2))
EVIDENCE["source_inspection"]=SOURCE_EVIDENCE

## 12. Thinking/reasoning disabling
Inspect Jinja's actual variables and compare the enabled/disabled rendered assistant prefixes. Only a supported local template switch whose disabled branch produces an empty, closed thinking block (or removes the thinking block) earns **THINKING MODE: DISABLED**. `**kwargs` alone is not evidence.

If no verifiable switch exists, use the direct-transcription prompt and report **UNVERIFIED**, never “disabled.” That fallback can run the one-page diagnostic but cannot unlock production under this notebook's strict no-thinking requirement. Do not strip reasoning and pass it off as OCR; unexpected thinking markers fail validation. No reasoning-enabled generation is performed.

In [ ]:
from jinja2 import Environment, meta
PROMPT = ("Transcribe all readable text visible in this scanned document image. Preserve the original language, script, "
          "line order, numbers, punctuation, handwriting, stamps, and MRZ characters. Do not translate, summarize, "
          "normalize, correct, infer, or guess. For unreadable text write [UNREADABLE]. Return only the transcription.")
def messages_for(image):
    return [{"role":"user","content":[{"type":"image","image":image},{"type":"text","text":PROMPT}]}]
try:
    template=processor.get_chat_template()
except (AttributeError,TypeError):
    template=getattr(processor,"chat_template",None) or getattr(tokenizer,"chat_template",None)
if isinstance(template,dict):
    if "default" not in template: stop("Multiple templates without an unambiguous default.")
    template=template["default"]
if not isinstance(template,str) or not template.strip(): stop("Local multimodal chat template is missing.")
print("LOCAL CHAT TEMPLATE:\n",template)
(OUT/"local_chat_template.jinja").write_text(template,encoding="utf-8")
try: template_vars=meta.find_undeclared_variables(Environment().parse(template))
except Exception:
    # Some HF templates use loopcontrols/extension-specific tags.
    template_vars=meta.find_undeclared_variables(Environment(extensions=["jinja2.ext.loopcontrols"]).parse(template))
print("Template variables:", sorted(template_vars))
THINK_KWARGS={}; THINKING_VERIFIED=False; THINKING_STATUS="UNVERIFIED — direct transcription prompt only"
def render_prompt(image, kwargs=None):
    return processor.apply_chat_template(messages_for(image),chat_template=template,tokenize=False,
                                         add_generation_prompt=True,**(THINK_KWARGS if kwargs is None else kwargs))
probe=Image.new("RGB",(64,64),"white")
# Names are considered only when the local template actually references them.
for key in ("enable_thinking","thinking"):
    if key not in template_vars: continue
    try:
        off=render_prompt(probe,{key:False}); on=render_prompt(probe,{key:True})
    except Exception as exc:
        print("Template switch rejected:",key,type(exc).__name__,str(exc)[:300]); continue
    common=0
    for a,b in zip(off,on):
        if a!=b: break
        common+=1
    changed_off=off[common:]; changed_on=on[common:]
    off_tail=off.rsplit("<think>",1)[-1] if "<think>" in off else ""
    empty_closed=bool(re.match(r"^\s*</think>\s*$",off_tail))
    removes_think="<think>" in changed_on and "<think>" not in changed_off and "</think>" not in changed_off
    if off!=on and (empty_closed or removes_think):
        THINK_KWARGS={key:False}; THINKING_VERIFIED=True
        THINKING_STATUS=f"DISABLED — local chat template {key}=False; branch difference verified"
        break
if THINKING_VERIFIED: print("THINKING MODE: DISABLED")
else: print("THINKING MODE:",THINKING_STATUS)
print("Mechanism:",THINKING_STATUS)
EVIDENCE["thinking"]={"status":THINKING_STATUS,"verified":THINKING_VERIFIED,"template_kwargs":THINK_KWARGS,"template_sha256":hashlib.sha256(template.encode()).hexdigest()}
persist_evidence()

## 13. Attention backend validation
Report loaded configuration and instantiated attention classes. These are distinct from actual executed kernels, which are captured on the first diagnostic generation if profiling is enabled. SDPA may dispatch to different kernels according to shapes/dtypes. Do not equate `flash-linear-attention` availability with FlashAttention activation.

In [ ]:
def attention_report():
    configs={"root":getattr(model.config,"_attn_implementation",None)}
    for name in ("text_config","vision_config"):
        c=getattr(model.config,name,None)
        if c is not None: configs[name]=getattr(c,"_attn_implementation",None)
    classes=Counter(type(m).__name__ for _,m in model.named_modules() if any(x in type(m).__name__.lower() for x in ("attention","gateddelta","linearattn")))
    return {"requested":REQUESTED_ATTN,"loaded_configuration":configs,"module_classes":dict(classes),"executed_kernels":"pending profiler"}
ATTENTION=attention_report()
print("ATTENTION BACKEND:",json.dumps(ATTENTION,indent=2))
EVIDENCE["attention"]=ATTENTION

## 14. Dataset extraction
Extract only relevant PDFs, preserving duplicate copies as independent documents. Both `CARTON SIGNATUTE` and the common spelling `CARTON SIGNATURE` map to the required type. Customer identifiers are the immediate parent directory of each PDF. Nested document subfolders cannot be inferred reliably and should be flattened upstream or inspected in inventory.

Reject path traversal, symlinks, duplicate ZIP paths, and archives exceeding the configured uncompressed-byte limit. A fresh extraction directory prevents stale files from contaminating inventory.

In [ ]:
def canonical_document(name):
    s=re.sub(r"\s+", " ",name.strip().upper())
    s=re.sub(r"\s*\(\d+\)(?=\.PDF$)","",s)
    s=s.replace("CARTON SIGNATURE.PDF","CARTON SIGNATUTE.PDF")
    return s if s in CFG.REQUIRED_DOCUMENTS else None
archive=Path(CFG.ZIP_PATH)
if not archive.is_file(): stop(f"Input ZIP missing: {archive.resolve()}")
EXTRACT=WORK/RUN_ID; EXTRACT.mkdir(parents=True,exist_ok=False)
CUSTOMER_DIRS=set(); seen=set()
with zipfile.ZipFile(archive) as z:
    infos=z.infolist()
    if sum(i.file_size for i in infos)>CFG.MAX_ZIP_UNCOMPRESSED_GIB*2**30: stop("ZIP exceeds uncompressed-size safety limit.")
    for info in infos:
        rel=Path(info.filename.replace("\\","/"))
        if rel.is_absolute() or ".." in rel.parts or (rel.parts and ":" in rel.parts[0]): stop("Unsafe ZIP member: "+info.filename)
        if stat.S_ISLNK(info.external_attr >> 16): stop("ZIP symlinks are not accepted.")
        target=(EXTRACT/rel).resolve()
        if not target.is_relative_to(EXTRACT): stop("ZIP member escapes extraction directory.")
        if not info.is_dir() and rel.suffix.lower()==".pdf": CUSTOMER_DIRS.add(rel.parent)
        if info.is_dir() or not canonical_document(rel.name): continue
        if target in seen: stop("Duplicate archive member path: "+info.filename)
        seen.add(target); target.parent.mkdir(parents=True,exist_ok=True)
        with z.open(info) as src,target.open("wb") as dst:
            import shutil
            shutil.copyfileobj(src,dst,1024*1024)
    # Retain empty leaf customer directories if represented explicitly in the archive.
    dirs={Path(i.filename.rstrip("/")) for i in infos if i.is_dir()}
    CUSTOMER_DIRS.update(d for d in dirs if not any(d in x.parents for x in dirs|CUSTOMER_DIRS) and d!=Path("."))
print("Relevant PDF copies extracted:",len(seen),"Candidate customer folders:",len(CUSTOMER_DIRS))

## 15. Document inventory
Inventory covers every detected customer, even if OCR is limited to one. “Readable” means PyMuPDF can open the PDF and access page objects; actual render/OCR errors remain page-level records. Password-protected and corrupted files are reported as unreadable. Duplicate filename copies remain separate rows. Inspect the folder-to-customer mapping before processing.

In [ ]:
INVENTORY=[]; ALL_PAGES=[]
for rel in sorted(CUSTOMER_DIRS,key=str):
    folder=EXTRACT/rel; customer=rel.name
    for required in CFG.REQUIRED_DOCUMENTS:
        matches=sorted([p for p in folder.glob("*") if p.is_file() and canonical_document(p.name)==required])
        if not matches:
            INVENTORY.append(dict(customer_id=customer,customer_folder=str(rel),document_type=required,document_name="",pdf_path="",presence="missing",readability="not_applicable",number_of_pages=0,error="")); continue
        for p in matches:
            count=0; readable="unreadable"; error=""
            try:
                with pymupdf.open(p) as pdf:
                    if pdf.needs_pass: raise ValueError("Password-protected PDF")
                    count=len(pdf)
                    if count==0: raise ValueError("PDF has zero pages")
                    for i in range(count): _=pdf[i].rect
                readable="readable"
                ALL_PAGES.extend(dict(customer_id=customer,customer_folder=str(rel),document_name=p.name,document_type=required,pdf_path=str(p),page_number=i+1,total_pages=count) for i in range(count))
            except Exception as exc:
                error=f"{type(exc).__name__}: {exc}"
                ERRORS.append({"customer_id":customer,"document_name":p.name,"pdf_path":str(p),"page_number":None,
                               "exception_type":type(exc).__name__,"message":str(exc),"traceback":traceback.format_exc()})
            INVENTORY.append(dict(customer_id=customer,customer_folder=str(rel),document_type=required,document_name=p.name,pdf_path=str(p),presence="present",readability=readable,number_of_pages=count,error=error))
def write_csv(path,rows):
    if not rows: return
    fields=list(dict.fromkeys(k for row in rows for k in row))
    with Path(path).open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader()
        for r in rows: w.writerow({k:json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list,tuple)) else v for k,v in r.items()})
write_csv(OUT/"document_inventory.csv",INVENTORY)
(OUT/"document_inventory.json").write_text(json.dumps(INVENTORY,ensure_ascii=False,indent=2),encoding="utf-8")
print("Inventory:",dict(Counter((r["presence"],r["readability"]) for r in INVENTORY)))
display(INVENTORY[:20])
if not ALL_PAGES: stop("No readable required PDF pages were found.")

## 16. Single PDF/page selection
Choose one seeded customer, then one readable PDF and one page. Automatic selection proves file readability, not visual legibility. Change the selection here if the displayed page is unsuitable. Other customers are only included when you explicitly increase `MAX_CUSTOMERS`.

In [ ]:
customers=sorted(set(r["customer_folder"] for r in ALL_PAGES))
if CFG.CUSTOMER_ID is not None:
    matched=[c for c in customers if Path(c).name==str(CFG.CUSTOMER_ID) or c==str(CFG.CUSTOMER_ID)]
    if len(matched)!=1: stop("CUSTOMER_ID missing or ambiguous; use the full relative customer folder.")
    chosen=matched[0]
else: chosen=random.Random(CFG.RANDOM_SEED).choice(customers)
selected_customers=[chosen]+[c for c in customers if c!=chosen]
if CFG.MAX_CUSTOMERS is not None: selected_customers=selected_customers[:CFG.MAX_CUSTOMERS]
PAGE_PLAN=[r for c in selected_customers for r in ALL_PAGES if r["customer_folder"]==c]
if CFG.MAX_PAGES is not None: PAGE_PLAN=PAGE_PLAN[:CFG.MAX_PAGES]
candidates=[r for r in ALL_PAGES if r["customer_folder"]==chosen]
if CFG.VALIDATION_PDF is not None:
    candidates=[r for r in candidates if r["document_name"]==CFG.VALIDATION_PDF or r["pdf_path"]==CFG.VALIDATION_PDF]
if not candidates: stop("Validation PDF selection matches no readable document.")
selected_pdf=candidates[0]["pdf_path"]
page_matches=[r for r in candidates if r["pdf_path"]==selected_pdf and r["page_number"]==CFG.VALIDATION_PAGE]
if not page_matches: stop("VALIDATION_PAGE is outside the selected PDF.")
SELECTED=page_matches[0]
SINGLE_PAGE_VALIDATION_PASSED=False
print("ONE PAGE SELECTED:",SELECTED)
print("Planned sequential dataset calls:",len(PAGE_PLAN),"Hard total call budget:",CFG.MAX_QWEN_CALLS)

## 17. PDF rendering and conservative preprocessing
Rendering automatically lowers effective DPI when a pathological page would exceed the render-pixel cap. Model input resizing preserves aspect ratio and does not upscale by default. Baseline preprocessing only converts to RGB and bounds resolution. Optional crop, rotation/deskew, contrast, grayscale, mild median denoise, and sharpening are explicit. Cropping may remove faint marks; compare visually before enabling it.

In [ ]:
def render_page(item, dpi=None):
    started=time.perf_counter()
    with pymupdf.open(item["pdf_path"]) as pdf:
        page=pdf[item["page_number"]-1]; rect=page.rect
        requested=dpi or CFG.RENDER_DPI
        scale=min(requested/72,math.sqrt(CFG.MAX_RENDER_PIXELS/max(1,rect.width*rect.height)))
        pix=page.get_pixmap(matrix=pymupdf.Matrix(scale,scale),colorspace=pymupdf.csRGB,alpha=False)
        im=Image.frombytes("RGB",(pix.width,pix.height),pix.samples)
    return im,{"pdf_width_points":rect.width,"pdf_height_points":rect.height,"effective_dpi":scale*72,
               "rendered_width":im.width,"rendered_height":im.height,"render_time":time.perf_counter()-started}
def prepare_image(im, long_edge=None, optional=None):
    started=time.perf_counter(); x=im.convert("RGB"); operations=[]
    optional=CFG.ENABLE_PREPROCESSING if optional is None else optional
    if optional:
        if CFG.ROTATE_DEGREES:
            if CFG.ROTATE_DEGREES%90: raise ValueError("Orientation must be a multiple of 90 degrees")
            x=x.rotate(CFG.ROTATE_DEGREES,expand=True,fillcolor="white"); operations.append("orientation")
        if CFG.DESKEW_DEGREES:
            if abs(CFG.DESKEW_DEGREES)>5: raise ValueError("Conservative deskew is limited to +/-5 degrees")
            x=x.rotate(CFG.DESKEW_DEGREES,Image.Resampling.BICUBIC,expand=True,fillcolor="white"); operations.append("deskew")
        if CFG.CROP_MARGINS:
            a=np.asarray(ImageOps.grayscale(x)); yy,xx=np.where(a<CFG.CROP_WHITE_THRESHOLD)
            if len(xx):
                pad=max(16,int(min(x.size)*.02)); box=(max(0,int(xx.min())-pad),max(0,int(yy.min())-pad),min(x.width,int(xx.max())+pad+1),min(x.height,int(yy.max())+pad+1))
                x=x.crop(box); operations.append("crop:"+str(box))
        if CFG.GRAYSCALE: x=ImageOps.grayscale(x).convert("RGB"); operations.append("grayscale")
        if CFG.CONTRAST!=1: x=ImageEnhance.Contrast(x).enhance(CFG.CONTRAST); operations.append("contrast")
        if CFG.DENOISE: x=x.filter(ImageFilter.MedianFilter(3)); operations.append("median3")
        if CFG.SHARPEN: x=x.filter(ImageFilter.UnsharpMask(radius=1,percent=60,threshold=3)); operations.append("mild_sharpen")
    edge=long_edge or CFG.MAX_IMAGE_LONG_EDGE
    if max(x.size)>edge: x.thumbnail((edge,edge),Image.Resampling.LANCZOS)
    if CFG.MIN_IMAGE_LONG_EDGE and max(x.size)<CFG.MIN_IMAGE_LONG_EDGE:
        factor=CFG.MIN_IMAGE_LONG_EDGE/max(x.size); x=x.resize((round(x.width*factor),round(x.height*factor)),Image.Resampling.LANCZOS)
    return x,{"preprocessing_time":time.perf_counter()-started,"preprocessing_operations":operations,"image_width":x.width,"image_height":x.height}
BASE_RENDER,RENDER_META=render_page(SELECTED)
PAGE_IMAGE,PREP_META=prepare_image(BASE_RENDER,optional=CFG.ENABLE_PREPROCESSING)
PREP_META["preprocessing_enabled"]=CFG.ENABLE_PREPROCESSING
print(RENDER_META,PREP_META)

## 18. Image visualization
Confirm text is visually legible and correctly oriented at the actual model-input resolution. The default first validation uses the baseline; enabling preprocessing requires rerunning this display and the validation cells. Optional transforms are evaluated separately. The saved PNG is the exact RGB input before processor resizing/patchification.

In [ ]:
display(PAGE_IMAGE)
PAGE_IMAGE.save(OUT/"single_page_input.png")
print("OCR PROMPT:\n"+PROMPT)

## 19. Vision-input validation
Instrument the image processor's real `preprocess` method to time vision preprocessing separately from template/tokenizer overhead. Inspect actual output tensor keys, shapes, dtypes, devices, finiteness, and nonconstant image content. Discover visual fields from the image processor's declared input names and output keys; require the model's forward path to accept them. Grid information is printed as exposed; no invented universal visual-token formula is used. Actual expanded `input_ids` length bounds the request.

In [ ]:
def tensor_report(batch):
    return {k:{"shape":list(v.shape),"dtype":str(v.dtype),"device":str(v.device)} if torch.is_tensor(v) else {"type":type(v).__name__} for k,v in batch.items()}
def make_inputs(image, verbose=False):
    started=time.perf_counter(); template_t=time.perf_counter()
    try: text=render_prompt(image)
    except Exception as exc: stop(f"Local chat template failed: {type(exc).__name__}: {exc}")
    template_seconds=time.perf_counter()-template_t
    vision_seconds=[0.0]; original=image_processor.preprocess
    def measured(*a,**kw):
        t=time.perf_counter()
        try: return original(*a,**kw)
        finally: vision_seconds[0]+=time.perf_counter()-t
    image_processor.preprocess=measured
    try:
        batch=processor(text=[text],images=[image],return_tensors="pt",padding=False)
    except Exception as exc: stop(f"Multimodal processor failed: {type(exc).__name__}: {exc}")
    finally: image_processor.preprocess=original
    duration=time.perf_counter()-started
    if "input_ids" not in batch: stop("Processor did not produce input_ids.")
    names=set(getattr(image_processor,"model_input_names",[]) or [])
    visual=[k for k,v in batch.items() if torch.is_tensor(v) and v.ndim>=2 and v.is_floating_point() and (k in names or any(s in k.lower() for s in ("pixel","image","vision","visual")))]
    if not visual: stop("No image/vision tensor generated. STOP before inference.")
    if any(not torch.isfinite(batch[k]).all().item() for k in visual): stop("Nonfinite vision tensor values.")
    forward_params=inspect.signature(model.forward).parameters
    accepts_kwargs=any(p.kind==inspect.Parameter.VAR_KEYWORD for p in forward_params.values())
    if not accepts_kwargs and any(k not in forward_params for k in visual): stop("Processor vision keys are not accepted by model.forward.")
    if batch["input_ids"].min()<0 or batch["input_ids"].max()>=embedding_size: stop("Out-of-range token IDs.")
    if batch["input_ids"].shape[-1]>CFG.MAX_INPUT_TOKENS: raise SafetyStop("Expanded multimodal input exceeds MAX_INPUT_TOKENS; reduce image resolution.")
    details={"processor_time":duration,"chat_template_time":template_seconds,"vision_preprocessing_time":vision_seconds[0],
             "text_processor_overhead_time":max(0,duration-template_seconds-vision_seconds[0]),"input_tokens":batch["input_ids"].shape[-1],
             "vision_keys":visual,"tensor_shapes":tensor_report(batch),"grid_info":{k:v.tolist() for k,v in batch.items() if "grid" in k and torch.is_tensor(v)},
             "vision_std":{k:float(batch[k].float().std()) for k in visual}}
    if verbose: print("Processor outputs:",json.dumps(details,indent=2,default=str))
    return batch,details,text
INPUTS,INPUT_META,RENDERED_PROMPT=make_inputs(PAGE_IMAGE,True)
print("RENDERED MULTIMODAL PROMPT:\n",RENDERED_PROMPT)
EVIDENCE["vision_input"]=INPUT_META

## 20. Single-page OCR test
Exactly one measured page inference is run here. The first call is **cold** and may include lazy kernel compilation. A separate warmup is offered later and is never included in production statistics. Generation uses a fresh deterministic configuration so hidden checkpoint sampling/beam settings cannot leak in, while preserving checkpoint EOS/PAD identifiers.

A repetition/time stopping criterion bounds damage; it never edits output. First-forward CUDA events estimate prefill; the remainder includes decoding **and generation-framework overhead**, not pure decoder-kernel time. Detailed first-page profiling adds overhead and is labelled accordingly. A forward hook verifies that the actual `generate` call forwards vision tensors, beyond their mere existence in the processor output.

In [ ]:
from transformers import GenerationConfig, StoppingCriteria, StoppingCriteriaList
import contextlib
class BoundedGeneration(StoppingCriteria):
    def __init__(self,start_length):
        self.start_length=start_length; self.started=time.perf_counter(); self.reason=None
    def __call__(self,input_ids,scores,**kwargs):
        seq=input_ids[:,self.start_length:]
        timed=time.perf_counter()-self.started>CFG.MAX_GENERATION_SECONDS
        repeated=seq.shape[-1]>=32 and bool((seq[:,-32:]==seq[:,-1:]).all(dim=1).any().item())
        if timed: self.reason="time_limit"
        elif repeated: self.reason="repeated_token_stop"
        return torch.full((input_ids.shape[0],),timed or repeated,dtype=torch.bool,device=input_ids.device)
eosc=model.generation_config.eos_token_id
if eosc is None: eosc=tokenizer.eos_token_id
EOS_IDS=[int(x) for x in eosc] if isinstance(eosc,(list,tuple)) else ([int(eosc)] if eosc is not None else [])
if not EOS_IDS: stop("No valid local EOS configuration.")
pad=model.generation_config.pad_token_id
if pad is None: pad=tokenizer.pad_token_id
if pad is None:
    pad=EOS_IDS[0]; print("PAD absent; using existing EOS as padding ID with explicit attention_mask.")
if any(x<0 or x>=embedding_size for x in EOS_IDS+[int(pad)]): stop("EOS/PAD IDs outside model vocabulary.")
GEN_CONFIG=GenerationConfig(max_new_tokens=CFG.MAX_NEW_TOKENS,do_sample=False,num_beams=1,use_cache=True,
    eos_token_id=EOS_IDS,pad_token_id=int(pad),bos_token_id=tokenizer.bos_token_id,
    repetition_penalty=1.0,return_dict_in_generate=False)
print("Generation configuration:",GEN_CONFIG.to_dict())
def diagnose_text(text,ids):
    compact="".join(text.split()); counts=Counter(compact); L=len(compact)
    dominant=max(counts.values(),default=0)/max(L,1)
    unique=len(counts)/max(L,1)
    ngrams=[compact[i:i+4] for i in range(max(0,L-3))]
    repeated_ngram=(1-len(set(ngrams))/len(ngrams)) if ngrams else 0.0
    diversity=len(set(ids))/max(len(ids),1)
    reasons=[]
    if L==0: reasons.append("empty")
    elif L<4: reasons.append("near_empty")
    if L>=12 and dominant>=CFG.DEGENERACY_THRESHOLD: reasons.append("dominant_character")
    if re.search(r"([^\s])\1{11,}",text): reasons.append("repeated_character_run")
    if len(ids)>=32 and diversity<0.08: reasons.append("low_token_diversity")
    if L>=80 and repeated_ngram>0.94: reasons.append("repeated_ngrams")
    if L>=12 and not any(c.isalnum() for c in compact) and compact!="[UNREADABLE]": reasons.append("punctuation_only")
    if re.search(r"<think>|</think>|<analysis>|</analysis>",text,re.I): reasons.append("reasoning_markers")
    return {"degenerate_output":bool(reasons),"degeneration_reason":";".join(reasons),"unique_character_ratio":unique,
            "dominant_character_ratio":dominant,"repeated_ngram_ratio":repeated_ngram,"token_diversity":diversity}
def infer(batch,meta,cap=None,profile=False,diagnostic=False):
    global CALLS
    if CALLS>=CFG.MAX_QWEN_CALLS: raise SafetyStop("MAX_QWEN_CALLS reached")
    bad=placement_report()
    if bad["offloaded_parameters"] or bad["non_cuda_large_buffers"]: stop("Device placement changed; unexpected offloading.")
    torch.cuda.synchronize(DEVICE); torch.cuda.reset_peak_memory_stats(DEVICE)
    transfer_t=time.perf_counter()
    # Preserve integer indices and processor float dtypes; model vision code controls its casts.
    batch={k:v.to(DEVICE) if torch.is_tensor(v) else v for k,v in batch.items()}
    torch.cuda.synchronize(DEVICE); transfer_time=time.perf_counter()-transfer_t
    if diagnostic: print("GPU inputs:",tensor_report(batch))
    prompt_len=batch["input_ids"].shape[-1]
    limit=cap or CFG.MAX_NEW_TOKENS
    guard=BoundedGeneration(prompt_len)
    pref_start=torch.cuda.Event(enable_timing=True); pref_end=torch.cuda.Event(enable_timing=True)
    state={"calls":0,"vision_forwarded":False,"first_complete":False}
    def before(module,args,kwargs):
        if state["calls"]==0:
            state["vision_forwarded"]=all(k in kwargs and torch.is_tensor(kwargs[k]) for k in meta["vision_keys"])
            pref_start.record()
        state["calls"]+=1
    def after(module,args,output):
        if state["calls"]==1: pref_end.record(); state["first_complete"]=True
    h1=model.register_forward_pre_hook(before,with_kwargs=True); h2=model.register_forward_hook(after)
    prof=None
    if profile:
        prof=torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA],record_shapes=False,profile_memory=False)
    CALLS+=1
    print(f"[QWEN START] call={CALLS} tokens_in={prompt_len} limit={limit} vision={meta['vision_keys']}")
    t=time.perf_counter()
    try:
        with torch.inference_mode(), (prof if prof is not None else contextlib.nullcontext()):
            seq=model.generate(**batch,generation_config=GEN_CONFIG,max_new_tokens=limit,
                stopping_criteria=StoppingCriteriaList([guard]))
            torch.cuda.synchronize(DEVICE)
        seconds=time.perf_counter()-t
    except torch.cuda.OutOfMemoryError:
        raise
    except Exception as exc:
        stop(f"Model generation/configuration failure: {type(exc).__name__}: {exc}")
    finally:
        h1.remove();h2.remove()
    if not state["vision_forwarded"]: stop("generate did not forward visual tensors into the model's first forward call.")
    decode_t=time.perf_counter()
    # This notebook accepts decoder-only multimodal models; no guessed encoder-decoder slicing.
    if getattr(model.config,"is_encoder_decoder",False): stop("Encoder-decoder output convention requires a separately validated adapter; not this Qwen path.")
    generated=seq[0,prompt_len:].detach().cpu().tolist()
    raw=tokenizer.decode(generated,skip_special_tokens=True,clean_up_tokenization_spaces=False)
    raw_special=tokenizer.decode(generated,skip_special_tokens=False,clean_up_tokenization_spaces=False)
    decode_time=time.perf_counter()-decode_t
    prefill=pref_start.elapsed_time(pref_end)/1000 if state["first_complete"] else None
    eos=bool(generated and generated[-1] in EOS_IDS)
    truncated=len(generated)>=limit and not eos
    diagnostics=diagnose_text(raw,generated)
    if re.search(r"<think>|</think>|<analysis>|</analysis>",raw_special,re.I):
        diagnostics["degenerate_output"]=True; diagnostics["degeneration_reason"]+=";reasoning_special_tokens"
    if guard.reason=="repeated_token_stop":
        diagnostics["degenerate_output"]=True; diagnostics["degeneration_reason"]+=";repeated_token_stop"
    if prof is not None:
        trace=OUT/"first_page_cuda_trace.json"; prof.export_chrome_trace(str(trace))
        events=prof.key_averages()
        relevant=[e.key for e in events if any(s in e.key.lower() for s in ("fp8","float8","scaled_mm","flash","attention","triton","gemm","delta"))]
        EVIDENCE["executed_ops"]=relevant[:100]
        EVIDENCE["fp8"]["kernel_status"]="Profiler captured operations; inspect trace for hardware kernel/dtype confirmation"
        print("Observed relevant profiler operations:",relevant[:40])
    result={"raw_text":raw,"generated_tokens":len(generated),"generated_token_ids":generated if diagnostic else None,
        "raw_decoded_with_special_tokens":raw_special if diagnostic else None,"cpu_to_gpu_time":transfer_time,
        "inference_time":seconds,"prefill_time_approx":prefill,"decode_generation_time_approx":max(0,seconds-prefill) if prefill is not None else None,
        "decode_time":decode_time,"tokens_per_second":len(generated)/max(seconds,1e-9),"gpu_memory":gpu_snapshot(),
        "truncated":truncated,"eos_seen":eos,"stop_reason":guard.reason,"vision_forwarded":state["vision_forwarded"],
        "profiling_enabled":profile,"status":"ok",**diagnostics}
    if truncated or diagnostics["degenerate_output"] or guard.reason: result["status"]="unhealthy"
    print(f"[QWEN END] tokens={len(generated)} seconds={seconds:.3f} tok/s={result['tokens_per_second']:.2f} status={result['status']}")
    del seq,batch
    return result
SINGLE_PAGE_VALIDATION_PASSED=False
PASSED_LEVELS.clear()
if "CACHE" in globals(): CACHE.clear()
FIRST=infer(INPUTS,INPUT_META,profile=CFG.PROFILE_FIRST_PAGE,diagnostic=True)
FIRST.update(SELECTED); FIRST.update(RENDER_META); FIRST.update(PREP_META); FIRST.update(INPUT_META)
FIRST["phase"]="first_cold_diagnostic"
FIRST["configuration_fingerprint"]=fingerprint()
FIRST["total_time"]=sum(FIRST[k] for k in ("render_time","preprocessing_time","processor_time","cpu_to_gpu_time","inference_time","decode_time"))
print("RAW TOKEN IDS:",FIRST["generated_token_ids"])
print("DECODED OUTPUT:\n",FIRST["raw_text"])
print("GENERATION LENGTH:",FIRST["generated_tokens"],"EOS:",FIRST["eos_seen"],"TRUNCATED:",FIRST["truncated"])
print("GPU MEMORY:",FIRST["gpu_memory"],"DEVICE:",PLACEMENT)
(OUT/"first_page_diagnostic.json").write_text(json.dumps(FIRST,ensure_ascii=False,indent=2),encoding="utf-8")
EVIDENCE["first_page"]=FIRST; persist_evidence()

## 21. Token degeneration analysis
Character/ngram/token heuristics are diagnostic flags, not an OCR confidence score. Long repeated MRZ fillers can trigger a false positive and should be reviewed, never automatically “repaired.” Inspect the original text and first tokens. Blank-image difference is evidence of sensitivity, not proof of accurate transcription.

In [ ]:
ids=FIRST["generated_token_ids"]
print("First generated tokens:",[(x,tokenizer.decode([x],skip_special_tokens=False)) for x in ids[:48]])
print("Most frequent generated IDs:",Counter(ids).most_common(12))
print("First 16 token diversity:",len(set(ids[:16])))
print("Degeneration:",FIRST["degenerate_output"],FIRST["degeneration_reason"])
print("Metrics:",{k:FIRST[k] for k in ("unique_character_ratio","dominant_character_ratio","repeated_ngram_ratio","token_diversity")})
TECHNICAL_SINGLE_OK=(FIRST["status"]=="ok" and FIRST["total_time"]<=CFG.MAX_PAGE_SECONDS)
ABLATION=None; WARMUP_TIME=None
if TECHNICAL_SINGLE_OK:
    # Separate short warmup; not an OCR quality test, never counted as a production page.
    warm= infer(INPUTS,INPUT_META,cap=16)
    WARMUP_TIME=warm["inference_time"]
    blank=Image.new("RGB",PAGE_IMAGE.size,"white")
    blank_inputs,blank_meta,_=make_inputs(blank)
    vision_different=any(INPUTS[k].shape!=blank_inputs[k].shape or not torch.equal(INPUTS[k],blank_inputs[k]) for k in INPUT_META["vision_keys"])
    blank_result=infer(blank_inputs,blank_meta,diagnostic=True)
    ABLATION={"vision_tensors_differ":vision_different,"raw_outputs_differ":FIRST["raw_text"]!=blank_result["raw_text"],
              "blank_result":blank_result,"interpretation":"Difference suggests visual sensitivity; manual transcription review is still required."}
    print("BLANK IMAGE OUTPUT:\n",blank_result["raw_text"])
    print("Ablation:",{k:v for k,v in ABLATION.items() if k!="blank_result"})
    EVIDENCE["ablation"]=ABLATION
else:
    print("First page unhealthy. Skipping additional inference; batch remains locked. Review saved evidence before retrying.")
EVIDENCE["warmup_seconds"]=WARMUP_TIME; persist_evidence()

## 22. Performance profiling and validation gate
The first measurement includes cold-start/profiler overhead; subsequent page timings are warm and unprofiled. CUDA event prefill includes the first forward (vision encoder + prompt processing). The remainder is approximate autoregressive generation plus framework overhead. Render/preprocess/processor/transfer/decode/write times are separate; do not add the nested processor breakdown twice.

**Review required:** compare the displayed image with its raw transcription. Enter a short exact excerpt actually visible on the page and present in the output, then set the review boolean. This is a quality gate, not a request for chain-of-thought. A nonempty output alone never unlocks batch. To enable transformed production input, compare section 24 first, set `CFG.ENABLE_PREPROCESSING=True`, then rerun sections 17–22 with the transformed input displayed and reviewed.


In [ ]:
# Edit only after visually checking the page against its output.
HUMAN_REVIEW_CONFIRMED = False
VISIBLE_TEXT_EXCERPT = ""

def refresh_validation():
    global SINGLE_PAGE_VALIDATION_PASSED,VALIDATED_FINGERPRINT
    text_ok=bool(VISIBLE_TEXT_EXCERPT.strip()) and VISIBLE_TEXT_EXCERPT in FIRST["raw_text"]
    checks={"single_page_healthy":TECHNICAL_SINGLE_OK,"thinking_disabled_verified":THINKING_VERIFIED,
            "image_forwarded":FIRST["vision_forwarded"],"ablation_image_changed":bool(ABLATION and ABLATION["vision_tensors_differ"]),
            "ablation_output_changed":bool(ABLATION and ABLATION["raw_outputs_differ"]),
            "human_review":HUMAN_REVIEW_CONFIRMED and text_ok,"configuration_unchanged":FIRST["configuration_fingerprint"]==fingerprint(),
            "preprocessing_validated":FIRST["preprocessing_enabled"]==CFG.ENABLE_PREPROCESSING}
    SINGLE_PAGE_VALIDATION_PASSED=all(checks.values())
    VALIDATED_FINGERPRINT=fingerprint() if SINGLE_PAGE_VALIDATION_PASSED else None
    if SINGLE_PAGE_VALIDATION_PASSED: PASSED_LEVELS.add(1)
    else: PASSED_LEVELS.clear()
    EVIDENCE["validation"]={"passed":SINGLE_PAGE_VALIDATION_PASSED,"checks":checks,"fingerprint":VALIDATED_FINGERPRINT}
    print("VALIDATION GATES:",checks); print("SINGLE_PAGE_VALIDATION_PASSED =",SINGLE_PAGE_VALIDATION_PASSED)
    persist_evidence()
def require_validation():
    if not SINGLE_PAGE_VALIDATION_PASSED or VALIDATED_FINGERPRINT!=fingerprint():
        raise SafetyStop("Batch locked: missing/failed validation or configuration changed. Revalidate the displayed single page.")
refresh_validation()
print("MODEL LOAD TIME:",MODEL_LOAD_TIME,"PROCESSOR LOAD TIME:",PROCESSOR_LOAD_TIME,"WARMUP TIME:",WARMUP_TIME)
print("FIRST MEASURED PAGE TIME:",FIRST["total_time"],"GENERATED TOKENS:",FIRST["generated_tokens"],"TOKENS/SECOND:",FIRST["tokens_per_second"])
print("50-page cold/profiled extrapolation (minutes):",FIRST["total_time"]*50/60)
print("Stage seconds:",{k:FIRST[k] for k in ("render_time","preprocessing_time","chat_template_time","vision_preprocessing_time","text_processor_overhead_time","cpu_to_gpu_time","prefill_time_approx","decode_generation_time_approx","decode_time")})

## 23. Image-resolution benchmark
Only run after technical single-page success and verified thinking control. Re-render the same page at resolutions corresponding to 1,000 and 1,800 pixels, then measure warm unprofiled OCR. Preserve and display each exact input and raw output. Defaults are not automatically changed: latency and output similarity cannot establish OCR accuracy. Choose the smallest visually satisfactory result, change the configuration, and revalidate before production.

In [ ]:
RESOLUTION_RESULTS=[]
if CFG.RUN_RESOLUTION_BENCHMARK and TECHNICAL_SINGLE_OK and THINKING_VERIFIED:
    for edge in CFG.RESOLUTION_LONG_EDGES:
        dpi=max(72,72*edge/max(RENDER_META["pdf_width_points"],RENDER_META["pdf_height_points"]))
        im,rm=render_page(SELECTED,dpi=dpi); im,pm=prepare_image(im,long_edge=edge,optional=False)
        batch,bm,_=make_inputs(im,True)
        rr=infer(batch,bm); rr.update(rm);rr.update(pm);rr.update(bm)
        rr["requested_edge"]=edge
        rr["total_time"]=sum(rr[k] for k in ("render_time","preprocessing_time","processor_time","cpu_to_gpu_time","inference_time","decode_time"))
        RESOLUTION_RESULTS.append(rr)
        im.save(OUT/f"resolution_{edge}.png");display(im)
        print("Edge:",edge,"page seconds:",rr["total_time"],"50-page estimated minutes:",rr["total_time"]*50/60)
        print(rr["raw_text"])
        if rr["status"]!="ok" or rr["total_time"]>CFG.MAX_PAGE_SECONDS:
            print("Resolution sweep stopped after unhealthy result."); break
    write_csv(OUT/"resolution_benchmark.csv",RESOLUTION_RESULTS)
else: print("Resolution benchmark skipped by configuration or failed technical/thinking checks.")
EVIDENCE["resolution_benchmark"]=RESOLUTION_RESULTS;persist_evidence()

## 24. Optional preprocessing comparison
Runs only when at least one optional operation is configured and the initial technical test succeeded. This diagnostic does not silently promote transforms to production. The original and transformed RGB images remain available. To validate transformed production input, set `CFG.ENABLE_PREPROCESSING=True` and rerun sections 17–22; the gate records that exact preprocessing mode.

In [ ]:
PREPROCESS_RESULT=None
operations_requested=any([CFG.ROTATE_DEGREES,CFG.DESKEW_DEGREES,CFG.CROP_MARGINS,CFG.GRAYSCALE,CFG.DENOISE,CFG.SHARPEN,CFG.CONTRAST!=1])
if operations_requested and TECHNICAL_SINGLE_OK and THINKING_VERIFIED:
    transformed,tm=prepare_image(BASE_RENDER,optional=True)
    display(PAGE_IMAGE);display(transformed)
    transformed.save(OUT/"preprocessed_comparison.png")
    tb,tmeta,_=make_inputs(transformed,True)
    PREPROCESS_RESULT=infer(tb,tmeta); PREPROCESS_RESULT.update(tm)
    print("Baseline:\n",FIRST["raw_text"],"\nTransformed:\n",PREPROCESS_RESULT["raw_text"])
    (OUT/"preprocessing_comparison.json").write_text(json.dumps(PREPROCESS_RESULT,ensure_ascii=False,indent=2),encoding="utf-8")
else: print("No optional operations requested, or diagnostic not healthy.")

## 25. Progressive 3 / 10 / 50-page benchmark
Explicitly set `CFG.BENCHMARK_LEVEL` to 2, 3, or 4 and rerun this cell after validation. Each level requires enough distinct pages and a successful previous level. Cached page results avoid repeated processing across levels. A failed page is recorded, safe PDF failures can continue within the level, and errors above tolerance block promotion. Degeneration, truncation, excessive latency, offloading, OOM, and excessive live-memory growth stop promotion immediately. Reserved-memory growth alone is normal allocator behaviour and does not prove a leak.

Results are appended to JSONL immediately for crash recovery. Human quality review of the initial page cannot guarantee the accuracy of all other pages.

In [ ]:
def page_key(item): return (item["pdf_path"],item["page_number"])
if "CACHE" not in globals(): CACHE={}
def save_page(record):
    start=time.perf_counter()
    record["write_time"]=None
    record["total_time"]=record.get("processing_time",record.get("total_time",0))
    with (OUT/"raw_ocr_results.jsonl").open("a",encoding="utf-8") as f:
        f.write(json.dumps(record,ensure_ascii=False,default=str)+"\n");f.flush()
    elapsed=time.perf_counter()-start
    record["write_time"]=elapsed; record["total_time"]+=elapsed
    # Separate append-only timing record avoids claiming that a JSON line can time its own completed write.
    with (OUT/"raw_ocr_write_times.jsonl").open("a",encoding="utf-8") as f:
        f.write(json.dumps({"pdf_path":record["pdf_path"],"page_number":record["page_number"],"write_time":elapsed,"total_time":record["total_time"]})+"\n")
    RESULTS.append(record)
def process_page(item,phase="production"):
    require_validation();start=time.perf_counter()
    result={**item,"configuration_fingerprint":fingerprint(),"phase":phase,"status":"error","error":"","raw_text":"","generated_tokens":0,
            "degenerate_output":False,"truncated":False}
    try:
        im,rm=render_page(item);im,pm=prepare_image(im)
    except Exception as exc:
        err={**item,"exception_type":type(exc).__name__,"message":str(exc),"traceback":traceback.format_exc()}
        ERRORS.append(err);result.update(error=err["message"],exception_type=err["exception_type"],traceback=err["traceback"])
        result["processing_time"]=time.perf_counter()-start;save_page(result);return result
    try:
        batch,bm,_=make_inputs(im)
        result.update(infer(batch,bm));result.update(rm);result.update(pm);result.update(bm)
    except torch.cuda.OutOfMemoryError as exc:
        err={**item,"exception_type":type(exc).__name__,"message":str(exc),"traceback":traceback.format_exc()}
        ERRORS.append(err);result.update(error=str(exc),exception_type=type(exc).__name__,traceback=err["traceback"])
        result["processing_time"]=time.perf_counter()-start;save_page(result)
        raise SafetyStop("CUDA OOM. Batch stopped; inspect VRAM and restart/revalidate at a smaller resolution.") from exc
    except (ConfigurationError,SafetyStop) as exc:
        result.update(error=str(exc),exception_type=type(exc).__name__,traceback=traceback.format_exc())
        result["processing_time"]=time.perf_counter()-start;save_page(result);raise
    result["processing_time"]=time.perf_counter()-start
    result["gpu_live_after_gib"]=torch.cuda.memory_allocated(DEVICE)/2**30
    save_page(result);return result

def run_level(level):
    require_validation()
    if level in PASSED_LEVELS: print("Level already passed:",level);return
    if level-1 not in PASSED_LEVELS: raise SafetyStop(f"Level {level-1} must pass first")
    target={2:3,3:10,4:50,5:len(PAGE_PLAN)}[level]
    if level==5 and (CFG.MAX_CUSTOMERS is not None or CFG.MAX_PAGES is not None):
        raise SafetyStop("Full dataset requires MAX_CUSTOMERS=None, MAX_PAGES=None, and rebuilding PAGE_PLAN in section 16.")
    if len(PAGE_PLAN)<target: raise SafetyStop(f"Level {level} needs {target} distinct planned pages; only {len(PAGE_PLAN)} available.")
    needed=sum(page_key(p) not in CACHE for p in PAGE_PLAN[:target])
    if CALLS+needed>CFG.MAX_QWEN_CALLS: raise SafetyStop("Insufficient Qwen call budget for this level.")
    records=[];initial_live=torch.cuda.memory_allocated(DEVICE)/2**30;failed=None
    for item in PAGE_PLAN[:target]:
        key=page_key(item)
        if key not in CACHE: CACHE[key]=process_page(item,phase=f"level_{level}")
        r=CACHE[key];records.append(r)
        if r.get("degenerate_output") or r.get("truncated") or r.get("stop_reason"):
            failed="Degenerate, truncated, or early-stopped output";break
        if r["total_time"]>CFG.MAX_PAGE_SECONDS: failed="Pathological page latency";break
        if r.get("gpu_live_after_gib",initial_live)-initial_live>CFG.MAX_LIVE_MEMORY_GROWTH_GIB:
            failed="Live CUDA memory grew beyond configured bound";break
        if sum(x["status"]=="error" for x in records)/len(records)>CFG.MAX_ERROR_RATE:
            failed="Error threshold exceeded";break
    report={"level":level,"target_pages":target,"observed_pages":len(records),"passed":failed is None and len(records)==target,
            "reason":failed,"mean_page_seconds":float(np.mean([r["total_time"] for r in records])) if records else None,"gpu":gpu_snapshot()}
    BENCHMARKS.append(report);EVIDENCE["progressive_benchmarks"]=BENCHMARKS;persist_evidence();print(report)
    if report["passed"]: PASSED_LEVELS.add(level)
    else: raise SafetyStop(f"Level {level} failed: {failed}")
if CFG.BENCHMARK_LEVEL>=2:
    for level in range(2,min(CFG.BENCHMARK_LEVEL,4)+1): run_level(level)
else: print("Progressive batch disabled: BENCHMARK_LEVEL=1. Validate first, then select level 2.")

## 26. Optional independent-page batching benchmark
Experimental only, after the 3-page sequential level passes. Compare sizes 1, 2, 4 against sequential references for the **same pages**, using left padding, separate prompts, and a list of independent images. Require exact greedy-output agreement as a conservative integrity check; disagreement is a reason to retain sequential processing, not proof that either output is correct. The experiment never changes the production path. Unsupported batch layout, OOM, or unhealthy outputs records the failure and retains batch size 1.

In [ ]:
BATCH_RESULTS=[]
def benchmark_batch(size,items,images,references):
    global CALLS
    require_validation()
    if CALLS>=CFG.MAX_QWEN_CALLS: raise SafetyStop("Qwen call budget exhausted")
    old_padding=tokenizer.padding_side; tokenizer.padding_side="left"
    old_pad=tokenizer.pad_token_id
    if old_pad is None: tokenizer.pad_token_id=int(pad)
    started=time.perf_counter()
    try:
        texts=[render_prompt(im) for im in images[:size]]
        b=processor(text=texts,images=images[:size],padding=True,return_tensors="pt")
    finally:
        tokenizer.padding_side=old_padding;tokenizer.pad_token_id=old_pad
    if b["input_ids"].shape[0]!=size: raise ValueError("Processor did not preserve independent batch examples")
    if b["input_ids"].shape[-1]>CFG.MAX_INPUT_TOKENS: raise ValueError("Batch input token cap exceeded")
    if not any(torch.is_tensor(v) and v.is_floating_point() and v.ndim>=2 for v in b.values()): raise ValueError("Batch vision tensor absent")
    processor_s=time.perf_counter()-started
    b={k:v.to(DEVICE) if torch.is_tensor(v) else v for k,v in b.items()}
    torch.cuda.synchronize(DEVICE);torch.cuda.reset_peak_memory_stats(DEVICE)
    plen=b["input_ids"].shape[-1];guard=BoundedGeneration(plen);CALLS+=1;t=time.perf_counter()
    with torch.inference_mode():
        seq=model.generate(**b,generation_config=GEN_CONFIG,stopping_criteria=StoppingCriteriaList([guard]))
    torch.cuda.synchronize(DEVICE);seconds=time.perf_counter()-t
    rows=[]
    for i in range(size):
        ids=seq[i,plen:].cpu().tolist()
        # Trim batch padding after first EOS; preserve text before it exactly.
        endpoint=next((j+1 for j,tok in enumerate(ids) if tok in EOS_IDS),len(ids))
        ids=ids[:endpoint];raw=tokenizer.decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=False)
        d=diagnose_text(raw,ids)
        rows.append({"raw_text":raw,**d,"truncated":len(ids)>=CFG.MAX_NEW_TOKENS and (not ids or ids[-1] not in EOS_IDS),"matches_sequential":raw==references[i]["raw_text"]})
    healthy=all(not r["degenerate_output"] and not r["truncated"] and r["matches_sequential"] for r in rows) and guard.reason is None
    return {"batch_size":size,"inference_seconds":seconds,"processor_seconds":processor_s,"latency_per_page":seconds/size,
            "pages_per_second":size/max(seconds,1e-9),"gpu":gpu_snapshot(),"healthy":healthy,"outputs":rows}
if CFG.ENABLE_BATCHING:
    require_validation()
    if 2 not in PASSED_LEVELS: raise SafetyStop("Pass the 3-page sequential benchmark before batching")
    count=min(4,len(PAGE_PLAN));items=PAGE_PLAN[:count];images=[];references=[]
    for item in items:
        key=page_key(item)
        if key not in CACHE: CACHE[key]=process_page(item,phase="batch_reference")
        r=CACHE[key]
        if r["status"]!="ok": raise SafetyStop("Unhealthy sequential batching reference")
        references.append(r);im,_=render_page(item);im,_=prepare_image(im);images.append(im)
    for size in (1,2,4):
        if size>count: continue
        if gpu_snapshot()["free_gib"]<CFG.MIN_FREE_VRAM_GIB: print("Insufficient free VRAM; retaining sequential mode");break
        try:
            report=benchmark_batch(size,items,images,references);BATCH_RESULTS.append(report);print(report)
            if not report["healthy"]: print("Batch integrity failed; fallback to size 1");break
        except ConfigurationError: raise
        except Exception as exc:
            BATCH_RESULTS.append({"batch_size":size,"healthy":False,"exception_type":type(exc).__name__,"error":str(exc),"traceback":traceback.format_exc()})
            print("Batch unsupported/unstable; fallback to size 1:",type(exc).__name__,str(exc)[:500]);break
    CFG.BATCH_SIZE=1
    (OUT/"batching_benchmark.json").write_text(json.dumps(BATCH_RESULTS,ensure_ascii=False,indent=2),encoding="utf-8")
else: print("Optional batching disabled. Production uses sequential page requests.")

## 27. Full RAW OCR pipeline
`BENCHMARK_LEVEL=5` requires successful 1/3/10/50-page levels, no customer/page caps, and a sufficient call budget. Rebuild the page plan after removing caps; previously completed pages are reused within this kernel. A dataset with fewer than 50 pages cannot satisfy this exact ladder: process it under the largest supported diagnostic level rather than claiming a completed 50-page validation. Full processing remains one page per request.

In [ ]:
if CFG.BENCHMARK_LEVEL==5:
    run_level(5)
else: print("Full-dataset execution disabled. Current benchmark level:",CFG.BENCHMARK_LEVEL)

## 28. Output persistence
JSONL is appended per page for recovery. Final exports below reconcile completed write timings and keep raw text unchanged. Special-token decoding and IDs are retained only for diagnostic pages, while normal `raw_text` is the continuation decoded without control tokens and without whitespace cleanup. CSV is quoted but untrusted OCR may be interpreted as a formula by spreadsheet software; import it as text. No fields are extracted or corrected.

In [ ]:
def export_results():
    # Include the original diagnostic even if the batch gate never opened.
    rows=RESULTS if RESULTS else [FIRST]
    tmp=OUT/"raw_ocr_results.jsonl.tmp"
    with tmp.open("w",encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False,default=str)+"\n")
    tmp.replace(OUT/"raw_ocr_results.jsonl")
    write_csv(OUT/"raw_ocr_results.csv",rows)
    with (OUT/"raw_ocr_results.txt").open("w",encoding="utf-8") as f:
        for r in rows:
            f.write(f"\n===== {r['customer_id']} | {r['document_name']} | page {r['page_number']}/{r['total_pages']} =====\n")
            f.write(r["raw_text"]);f.write("\n")
    perf=[{k:v for k,v in r.items() if k not in ("raw_text","raw_decoded_with_special_tokens","generated_token_ids","traceback")} for r in rows]
    write_csv(OUT/"raw_ocr_performance.csv",perf)
    (OUT/"page_errors.json").write_text(json.dumps(ERRORS,ensure_ascii=False,indent=2),encoding="utf-8")
    (OUT/"progressive_benchmarks.json").write_text(json.dumps(BENCHMARKS,indent=2),encoding="utf-8")
    persist_evidence();print("Saved:",OUT)
export_results()

## 29. Performance and diagnostic summary
Production estimates use successful warm sequential pages only. The cold first diagnostic, warmup, ablation, resolution experiments, and batch experiments are excluded. When no production page has run, only a clearly labelled first-page extrapolation is available.

In [ ]:
successful=[r for r in RESULTS if r["status"]=="ok" and r.get("configuration_fingerprint")==fingerprint()]
if successful:
    times=np.array([r["total_time"] for r in successful]);secs=sum(r["inference_time"] for r in successful)
    performance={"successful_pages":len(successful),"mean_page_seconds":float(times.mean()),"median_page_seconds":float(np.median(times)),
        "p95_page_seconds":float(np.quantile(times,.95)),"50_page_estimate_minutes":float(times.mean()*50/60),
        "aggregate_generation_tokens_per_second":sum(r["generated_tokens"] for r in successful)/max(secs,1e-9)}
else: performance={"successful_production_pages":0,"cold_profiled_50_page_estimate_minutes":FIRST["total_time"]*50/60}
print("PERFORMANCE:",json.dumps(performance,indent=2))
summary={"Model":str(MODEL_PATH),"Model class":type(model).__name__,"Processor":type(processor).__name__,
    "Multimodal support":True,"Vision input detected":FIRST["vision_forwarded"],"FP8 status":EVIDENCE["fp8"],
    "Device":str(DEVICE),"CPU offloading":bool(PLACEMENT["offloaded_parameters"]),"Attention backend":ATTENTION,
    "Thinking/reasoning":THINKING_STATUS,"Image dimensions":PAGE_IMAGE.size,"Visual input shape":INPUT_META["tensor_shapes"],
    "Input tokens":FIRST["input_tokens"],"Generated tokens":FIRST["generated_tokens"],"Inference time":FIRST["inference_time"],
    "Tokens/sec":FIRST["tokens_per_second"],"GPU allocated":FIRST["gpu_memory"]["allocated_gib"],"GPU peak":FIRST["gpu_memory"]["peak_allocated_gib"],
    "Output degenerate":FIRST["degenerate_output"],"Output truncated":FIRST["truncated"],"Single-page gate":SINGLE_PAGE_VALIDATION_PASSED}
print("="*50+"\nQWEN OCR DIAGNOSTIC\n"+json.dumps(summary,ensure_ascii=False,indent=2,default=str)+"\n"+"="*50)
EVIDENCE["performance"]=performance;EVIDENCE["summary"]=summary;persist_evidence()

## 30. Evidence-based root-cause report
Separate observed failures, evidence against a hypothesis, and unresolved questions. A clean redesigned run cannot reconstruct the cause of the previous notebook's failure. Visual tensors reaching the forward pass plus changed ablation outputs supports an operational vision path; only comparison with the source page supports transcription accuracy. Slow eager attention, an FP8 module class, and resolution sensitivity are clues, not automatic causal conclusions.

In [ ]:
def root_cause_report():
    report=[]
    def add(hypothesis,assessment,evidence): report.append({"hypothesis":hypothesis,"assessment":assessment,"evidence":evidence})
    add("Image missing", "Evidence against" if FIRST["vision_forwarded"] else "Observed failure", {"keys":INPUT_META["vision_keys"],"forwarded":FIRST["vision_forwarded"],"ablation":None if not ABLATION else {k:v for k,v in ABLATION.items() if k!="blank_result"}})
    add("Incorrect architecture", "Evidence against in this run", {"config_class":type(hf_config).__name__,"model_class":type(model).__name__,"vision_config":bool(vision_config)})
    add("CPU offloading", "Evidence against in this run", PLACEMENT)
    add("FP8 misconfiguration", "Quantizer loaded; kernel efficiency/provenance not conclusively established", EVIDENCE["fp8"])
    add("Excessive image resolution / visual tokens", "Measured; evaluate sweep and visual quality before assigning causality", {"dimensions":PAGE_IMAGE.size,"tokens":INPUT_META["input_tokens"],"grids":INPUT_META["grid_info"],"sweep":[{"edge":r["requested_edge"],"seconds":r["total_time"],"input_tokens":r["input_tokens"]} for r in RESOLUTION_RESULTS]})
    add("Reasoning overhead", "Disable mechanism verified" if THINKING_VERIFIED else "Unresolved; batch blocked", EVIDENCE["thinking"])
    add("Excessive generation length", "Observed cap reached" if FIRST["truncated"] else "No cap exhaustion on first page", {"tokens":FIRST["generated_tokens"],"limit":CFG.MAX_NEW_TOKENS,"eos":FIRST["eos_seen"],"stop":FIRST["stop_reason"]})
    add("Token repetition degeneration", "Observed failure" if FIRST["degenerate_output"] else "Not detected on first page", FIRST["degeneration_reason"])
    add("Attention inefficiency", "Unresolved without matched backend benchmark", {"configuration":ATTENTION,"observed_ops":EVIDENCE.get("executed_ops",[])})
    fraction=FIRST["processor_time"]/max(FIRST["total_time"],1e-9)
    add("Processor bottleneck", "Large measured share" if fraction>.3 else "Small measured share on first page", {"fraction_of_page_time":fraction,"processor_seconds":FIRST["processor_time"],"vision_seconds":FIRST["vision_preprocessing_time"]})
    add("Historical >3-hour runtime", "Cannot identify historical cause from a replacement run alone", {"previous_average_seconds_per_page":216,"current_first_page_seconds":FIRST["total_time"],"cold_and_profiled":FIRST["profiling_enabled"],"performance":EVIDENCE.get("performance")})
    add("Tokenizer mismatch / file corruption", "Basic checks passed; provenance unresolved", "Header/index bounds and vocab range checked. Authenticity and numerical checkpoint correctness require trusted reference checksums or a known-good local deployment.")
    (OUT/"root_cause_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    lines=["# Qwen RAW OCR root-cause report", "", "Historical causality remains unproven unless supported by matched observations.", ""]
    for r in report:
        lines.extend([f"## {r['hypothesis']}",r["assessment"],"",json.dumps(r["evidence"],ensure_ascii=False,default=str),""])
        print(r["hypothesis"],"—",r["assessment"],"—",r["evidence"])
    (OUT/"root_cause_report.md").write_text("\n".join(lines),encoding="utf-8")
    return report
ROOT_CAUSE_REPORT=root_cause_report()

## 31. Final recommendations and operating procedure
1. If architecture, processor, FP8, or device checks stop execution, repair that exact incompatibility before rerunning. Do not increase the token limit or insert a guessed model class.
2. If the first result repeats punctuation, inspect token IDs, processor tensors, actual forward inputs, EOS, quantizer and the profiler trace. This notebook preserves the failed evidence and blocks batch.
3. If thinking control is unverified, inspect the printed local template and reviewed model source. Do not assert it is disabled merely because the prompt requests transcription. The notebook intentionally refuses to promise a universal adapter for unknown custom code.
4. Compare the displayed original with the transcription and record the review in section 22. After that gate passes, set `BENCHMARK_LEVEL=2`, then 3, then 4. Keep the kernel and model loaded.
5. Use the resolution and preprocessing comparisons to choose a visually adequate input. Changing production image/generation settings invalidates the gate and requires a new measured test; rerun the single-page preparation/test/ablation/review cells, not the model-loading cells.
6. Treat truncation as incomplete OCR. Inspect dense pages and consider a separately validated page-region strategy later; this baseline never silently increases to thousands of tokens or adds a fallback engine.
7. Use warm measured pages for the 50-page estimate. Trace overhead and cold compilation can inflate the first call. Review peak and live memory; cached reserved VRAM alone is not a leak. No per-page `empty_cache()` calls are used.
8. Keep independent-page batching experimental. Production deliberately stays sequential after the experiment, even if a larger batch succeeds.

### Dependency/API references (documentation only; not accessed by notebook)
- [Hugging Face multimodal chat templates](https://huggingface.co/docs/transformers/main/chat_templating_multimodal)
- [Writing and inspecting local chat templates](https://huggingface.co/docs/transformers/chat_templating_writing)
- [Fine-grained FP8 implementation documentation/source](https://github.com/huggingface/transformers/blob/main/docs/source/en/quantization/finegrained_fp8.md)

Installed local source/configuration is authoritative for this notebook. No claim is made that a directory named Qwen3.6 is supported by every Transformers release. The notebook itself makes no network calls.
